# AI Trader - Qwen2.5-7B LoRA 微调

用A股历史数据微调 Qwen 模型，让它学会预测股票走势概率分布

**运行前**: 修改 → 笔记本设置 → GPU (T4)

In [ ]:
# 1. 检查GPU
!nvidia-smi

In [ ]:
# 2. 安装依赖（约3分钟）
!pip install -q transformers peft datasets accelerate bitsandbytes

In [ ]:
# 3. 上传训练数据
from google.colab import files
import zipfile, os

print('请上传 colab_upload.zip:')
uploaded = files.upload()

# 解压
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('.')
        print(f'解压完成: {fname}')

# 验证
print(f'\n训练样本: {sum(1 for _ in open("finetune_data/train.jsonl"))}')
print(f'验证样本: {sum(1 for _ in open("finetune_data/val.jsonl"))}')

In [ ]:
# 4. 微调（核心步骤，约1-2小时）
# 如果T4显存不够，把 batch_size 改成 2
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "./qwen-stock-lora"
DATA_DIR = "./finetune_data"

# ---- 加载模型（首次下载约14GB，之后有缓存）----
print('加载基础模型...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.enable_input_require_grads()

# ---- 应用LoRA ----
print('应用LoRA...')
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---- 加载数据 ----
print('加载数据...')
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(f'{DATA_DIR}/train.jsonl')
val_data = load_jsonl(f'{DATA_DIR}/val.jsonl')
print(f'Train: {len(train_data)}, Val: {len(val_data)}')

# ---- 预处理 ----
def preprocess(messages_list):
    inputs = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for messages in messages_list:
        full = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=False)
        full_tok = tokenizer(full, truncation=True, max_length=1024)
        prompt_len = len(tokenizer(prompt, truncation=True, max_length=1024)['input_ids'])
        labels = [-100]*prompt_len + full_tok['input_ids'][prompt_len:]
        inputs['input_ids'].append(full_tok['input_ids'])
        inputs['attention_mask'].append(full_tok['attention_mask'])
        inputs['labels'].append(labels)
    return inputs

train_ds = Dataset.from_list(train_data).map(lambda x: preprocess(x['messages']), batched=True, remove_columns=['messages'])
val_ds = Dataset.from_list(val_data).map(lambda x: preprocess(x['messages']), batched=True, remove_columns=['messages'])

# ---- 训练 ----
print('开始训练...')
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,   # T4用2，A100可以用4
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # 等效batch=16
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

result = trainer.train()
print(f'\n训练完成! Loss: {result.training_loss:.4f}')

# 保存
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'模型已保存: {OUTPUT_DIR}')

In [ ]:
# 5. 测试：看模型预测效果
import json, torch

model.eval()

# 加载测试集前5条
test_data = load_jsonl(f'{DATA_DIR}/test.jsonl')[:5]

print('='*60)
print('模型预测 vs 实际结果')
print('='*60)

for i, messages in enumerate(test_data):
    input_msgs = messages[:-1]
    expected = json.loads(messages[-1]['content'])
    
    text = tokenizer.apply_chat_template(input_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)
    
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    print(f'\n--- 样本 {i+1} ---')
    print(f'实际: 1d={expected["actual_1d"]:+.2f}%  1w={expected["actual_1w"]:+.2f}%  1m={expected["actual_1m"]:+.2f}%')
    
    try:
        pred = json.loads(response.strip())
        if '1d' in pred:
            p = pred['1d']
            print(f'预测1d: up={p.get("up",0):.2f} flat={p.get("flat",0):.2f} down={p.get("down",0):.2f}')
        if '1w' in pred:
            p = pred['1w']
            print(f'预测1w: up={p.get("up",0):.2f} flat={p.get("flat",0):.2f} down={p.get("down",0):.2f}')
    except:
        print(f'模型输出(未解析): {response[:200]}')

In [ ]:
# 6. 下载微调好的模型
import shutil
from google.colab import files as colab_files

shutil.make_archive('qwen-stock-lora', 'zip', '.', 'qwen-stock-lora')
print('下载模型...')
colab_files.download('qwen-stock-lora.zip')
print('下载完成后，解压到 ai-trader/qwen-stock-lora/ 目录')